# Side-Channel Trace Acquisition for Leakage Assessment

This notebook implements an automated side-channel trace acquisition framework using the ChipWhisperer platform. It supports fixed-versus-random input experiments across different firmware implementations, enabling the collection of power traces and associated metadata for subsequent statistical leakage assessment. The acquisition procedure includes optional input masking and externally supplied randomness, with experimental data stored in HDF5 format.

## Experimental Configuration

The following parameters define the hardware environment, firmware selection, and trace acquisition settings:

- **Hardware platform:** `SCOPETYPE` specifies the acquisition interface (`OPENADC`), while `PLATFORM` identifies the target microcontroller (`CW308_STM32F4`).
- **Firmware configuration:** `CRYPTO_TARGET` and `SS_VER` define the cryptographic target (`TINYAES128C`) and SimpleSerial protocol version (`SS_VER_1_1`), respectively.
- **Implementation selection:** `base_dir_hex` specifies the firmware directory, while `masking_list` selects the implementations to be evaluated.
- **Dataset organization:** `base_dir_traces` specifies the output directory, and `experiment_name` provides the dataset identifier. Acquisition results are organized into implementation-specific subdirectories.
- **Acquisition bounds:** `iteration_start` and `iteration_end` define the iteration range, with each iteration comprising 1,000 trace acquisitions.
- **Trace dimensions:** `TRACES_PER_FILE` determines the number of traces stored per HDF5 file (1,000), while `SAMPLES_PER_TRACE` specifies the number of ADC samples per trace.
- **Trace composition:** `iterations_per_full_trace` specifies the intended number of sub-traces concatenated to form a complete execution trace, depending on the implementation.

These parameters provide a centralized configuration for controlling acquisition conditions and organizing the resulting experimental datasets.

In [ ]:
import os
from tqdm.notebook import trange
import numpy as np
import time
import secrets

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F4'
CRYPTO_TARGET = 'TINYAES128C'
SS_VER = 'SS_VER_1_1'

# Firmware implementations
base_dir_hex = ".\\..\\..\\firmware\\mcu"

original_implementation_folder =  "OriginalImplementation"
sanitized_implementation_folder = "SanitizedImplementation"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% DEFINE PARAMETERS HERE %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
base_dir_traces = "path/to/traces/folder" 

experiment_name = "ExperimentName"                   # Dataset subfolder name

masking_list = [original_implementation_folder]                   # List of firmware implementations to evaluate

iteration_start = 0                                            # Index of the first trace batch
iteration_end = 10000000                                       # Index of the last trace batch

TRACES_PER_FILE = 1000
SAMPLES_PER_TRACE = 300000

iterations_per_full_trace = 1                          # Number of concatenated sub-traces per full trace (3 for unmasked, 11 for masked)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


## Acquisition Platform Initialization

The ChipWhisperer acquisition environment is initialized through the shared setup script, establishing communication with the target microcontroller and configuring the interfaces required for firmware programming and side-channel measurements.

In [ ]:
# CHIPWHISPERER-LITE CW308 PLATFORM INITIALIZATION
%run "../Setup_Scripts/Setup_Generic.ipynb"

## Target Firmware Compilation

The cryptographic firmware is compiled for the selected microcontroller platform using the specified cryptographic target and SimpleSerial protocol version. This step generates the executable firmware required for subsequent trace acquisition.

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"

cd ../../firmware/mcu/OriginalImplementation
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

# cd ../../firmware/mcu/SanitizedImplementation
# make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

## Side-Channel Trace Acquisition and Storage

This cell implements the trace acquisition procedure using a balanced fixed-versus-random input distribution with shuffled class labels. For each execution, optional external randomness and input masking are applied before capturing the corresponding side-channel waveform. Traces and associated metadata, including plaintexts, masks, seeds, and class labels, are stored in sequential HDF5 files for subsequent leakage assessment.

In [ ]:
import os
import h5py
import numpy as np
from tqdm import tqdm
import secrets

LABEL_FIXED = 0
LABEL_RANDOM = 1
LABEL_SHUFFLE_SEED = None

def open_hdf5_file(file_index, experiment_name, base_dir_traces):
    file_path = os.path.join(base_dir_traces, experiment_name, f"{experiment_name}_traces_part{file_index:03d}.h5")
    f = h5py.File(file_path, "w")
    dset_traces = f.create_dataset("traces", shape=(TRACES_PER_FILE, SAMPLES_PER_TRACE), dtype="uint16")
    dset_plaintexts = f.create_dataset("plaintexts", shape=(TRACES_PER_FILE, 16), dtype="uint8")
    dset_masks = f.create_dataset("masks", shape=(TRACES_PER_FILE, 16), dtype="uint8")
    dset_seeds = f.create_dataset("seeds", shape=(TRACES_PER_FILE, 32), dtype="uint8")
    dset_labels = f.create_dataset("labels", shape=(TRACES_PER_FILE,), dtype="uint8")
    dset_labels.attrs["0"] = "fixed"
    dset_labels.attrs["1"] = "random"
    return f, dset_traces, dset_plaintexts, dset_masks, dset_seeds, dset_labels

def generate_shuffled_labels(n_traces, rng):
    n_fixed = n_traces // 2
    n_random = n_traces - n_fixed
    labels = np.concatenate((np.full(n_fixed, LABEL_FIXED, dtype=np.uint8), np.full(n_random, LABEL_RANDOM, dtype=np.uint8)))
    rng.shuffle(labels)
    return labels

def capture_traces(use_seed, experiment_name):
    scope.gain.db = 1
    scope.adc.samples = SAMPLES_PER_TRACE
    scope.adc.offset = 0
    scope.adc.timeout = 2
    target.baud = 38400
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    fixed_plaintext = bytes([108, 81, 56, 195, 111, 10, 64, 204, 0, 43, 197, 36, 103, 202, 212, 146])
    label_rng = np.random.default_rng(LABEL_SHUFFLE_SEED)
    os.makedirs(os.path.join(base_dir_traces, experiment_name), exist_ok=True)
    file_index = 2394
    trace_index = 0
    f, dset_traces, dset_plaintexts, dset_masks, dset_seeds, dset_labels = open_hdf5_file(file_index, experiment_name, base_dir_traces)
    progress = tqdm(total=TRACES_PER_FILE, desc=f"File {file_index:03d}", unit="trace")
    
    try:
        for iteration in range(iteration_start, iteration_end):
            labels = generate_shuffled_labels(1000, label_rng)

            for i in range(1000):
                label = labels[i]

                if label == LABEL_FIXED:
                    text = fixed_plaintext
                else:
                    key, text = ktp.next()

                if len(text) != 16:
                    raise RuntimeError(f"Unexpected plaintext length: {len(text)}")

                if use_seed:
                    external_seed = [int.from_bytes(secrets.token_bytes(1), "little") for _ in range(32)]

                    for k in range(2):
                        target.simpleserial_write("s", external_seed[k * 16:(k + 1) * 16])
                        response = target.simpleserial_read("r", 16)

                        if response is None:
                            raise RuntimeError("No response while sending seed")

                    mask = [int.from_bytes(secrets.token_bytes(1), "little") for _ in range(16)]
                    masked_text = bytes(m ^ t for m, t in zip(mask, text))
                    target.simpleserial_write("k", mask)
                    response = target.simpleserial_read("r", 16)

                    if response is None:
                        raise RuntimeError("No response while sending mask")

                    scope.adc.offset = 0
                    trace = cw.capture_trace(scope, target, masked_text, as_int = True)

                else:
                    mask = [0] * 16
                    external_seed = [0] * 32
                    scope.adc.offset = 0
                    trace = cw.capture_trace(scope, target, text, as_int = True)

                if trace is None:
                    raise RuntimeError(f"Capture failed at iteration {iteration}, trace {i}")

                if len(trace.wave) != SAMPLES_PER_TRACE:
                    raise RuntimeError(f"Unexpected trace length: {len(trace.wave)} (expected {SAMPLES_PER_TRACE})")

                dset_traces[trace_index, :] = trace.wave
                dset_plaintexts[trace_index, :] = np.frombuffer(text, dtype=np.uint8)
                dset_masks[trace_index, :] = np.asarray(mask, dtype=np.uint8)
                dset_seeds[trace_index, :] = np.asarray(external_seed, dtype=np.uint8)
                dset_labels[trace_index] = label
                trace_index += 1
                progress.update(1)
                
                # print(f"-----TRACE {i}-----")
                # print("Plaintext: ", text)
                # for j in range(25):
                #     target.simpleserial_write('r',[j, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
                #     msg_res = target.simpleserial_read('m', 8)
                #     print("Lane ", j,": ",' '.join(f'{b:02X}' for b in msg_res))
                # print("\n\n")

                # ---------CLOCK CYCLES COUNT---------
                # target.simpleserial_write('c',[0])
                # num_cycles = int.from_bytes(target.simpleserial_read('r', 4), byteorder="little")
                # print("Number of clock cycles: ", num_cycles)
                # print("ADC Frequency: ", scope.clock.adc_freq)


                if trace_index >= TRACES_PER_FILE:
                    f.flush()
                    f.close()
                    progress.close()
                    file_index += 1
                    trace_index = 0
                    is_last_iteration = iteration == iteration_end - 1
                    is_last_trace = i == 999

                    if not (is_last_iteration and is_last_trace):
                        f, dset_traces, dset_plaintexts, dset_masks, dset_seeds, dset_labels = open_hdf5_file(file_index, experiment_name, base_dir_traces)
                        progress = tqdm(total=TRACES_PER_FILE, desc=f"File {file_index:03d}", unit="trace")
                    else:
                        f = None
                        progress = None

    finally:
        if progress is not None:
            progress.close()

        if f is not None:
            try:
                f.flush()
            except Exception:
                pass

            try:
                f.close()
            except Exception:
                pass

## Firmware Deployment and Experimental Execution

Each selected firmware implementation is programmed onto the target microcontroller before initiating the corresponding acquisition procedure. The resulting datasets are stored in implementation-specific directories, enabling separate leakage assessments across the evaluated implementations.

In [ ]:
for masking in masking_list:
    masking_path = os.path.normpath(os.path.join(base_dir_hex, masking))
    for filename in os.listdir(masking_path):
        if filename.endswith(".hex"):
            hex_path = os.path.normpath(os.path.join(masking_path, filename))
    experiment_name = ("exp_" + masking)
    print(f"Experiment: {experiment_name}")
    cw.program_target(scope, prog, hex_path)
    capture_traces(use_seed=True, experiment_name=experiment_name)                    